# ELOM OLUCHI LUCY
# Track: Data Analytics (Oasis Infobyte SIP)
# Task: Level 1 - Cleaning Data

In [1]:
import pandas as pd 
df = pd.read_excel("dirty_cafe_sales.xlsx")
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2,4,Credit Card,Takeaway,45177
1,TXN_4977031,Cake,4,3,12,Cash,In-store,45062
2,TXN_4271903,Cookie,4,1,ERROR,Credit Card,In-store,45126
3,TXN_7034554,Salad,2,5,10,UNKNOWN,UNKNOWN,45043
4,TXN_3160411,Coffee,2,2,4,Digital Wallet,In-store,45088


In [2]:
print(df.shape)
print(df.isna().sum())
print("Duplicates:", df.duplicated().sum())

(10000, 8)
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64
Duplicates: 0


In [3]:
print((df == "ERROR").sum())
print((df == "UNKNOWN").sum())
print(df.dtypes)

Transaction ID        0
Item                292
Quantity            170
Price Per Unit      190
Total Spent         164
Payment Method      306
Location            358
Transaction Date    142
dtype: int64
Transaction ID        0
Item                344
Quantity            171
Price Per Unit      164
Total Spent         165
Payment Method      293
Location            338
Transaction Date    159
dtype: int64
Transaction ID         str
Item                   str
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method         str
Location               str
Transaction Date    object
dtype: object


In [4]:
import numpy as np
df_raw = df.copy()
df = df.replace(["ERROR", "UNKNOWN"], np.nan)
print(df.isna().sum())

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


In [5]:
for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    df[col] = pd.to_numeric(df[col])

df["Transaction Date"] = pd.to_datetime(
    pd.to_numeric(df["Transaction Date"]), unit="D", origin="1899-12-30"
)

print(df.dtypes)
print(df.isna().sum())
print(df["Transaction Date"].min(), df["Transaction Date"].max())

Transaction ID                str
Item                          str
Quantity                  float64
Price Per Unit            float64
Total Spent               float64
Payment Method                str
Location                      str
Transaction Date    datetime64[s]
dtype: object
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64
2023-01-01 00:00:00 2023-12-31 00:00:00


In [6]:
for col in ["Item", "Payment Method", "Location"]:
    print(df[col].value_counts(dropna=False))
    print()

Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
NaN          969
Name: count, dtype: int64

Payment Method
NaN               3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64

Location
NaN         3961
Takeaway    3022
In-store    3017
Name: count, dtype: int64



In [7]:
# 1. Missing Total Spent = Quantity x Price Per Unit
m = df["Total Spent"].isna() & df["Quantity"].notna() & df["Price Per Unit"].notna()
print("Total Spent recovered:", m.sum())
df.loc[m, "Total Spent"] = df.loc[m, "Quantity"] * df.loc[m, "Price Per Unit"]

# 2. Missing Quantity = Total Spent / Price Per Unit
m = df["Quantity"].isna() & df["Total Spent"].notna() & df["Price Per Unit"].notna()
print("Quantity recovered:", m.sum())
df.loc[m, "Quantity"] = df.loc[m, "Total Spent"] / df.loc[m, "Price Per Unit"]

# 3. Missing Price Per Unit = Total Spent / Quantity
m = df["Price Per Unit"].isna() & df["Total Spent"].notna() & df["Quantity"].notna()
print("Price Per Unit recovered:", m.sum())
df.loc[m, "Price Per Unit"] = df.loc[m, "Total Spent"] / df.loc[m, "Quantity"]

print(df[["Quantity", "Price Per Unit", "Total Spent"]].isna().sum())

Total Spent recovered: 462
Quantity recovered: 441
Price Per Unit recovered: 495
Quantity          38
Price Per Unit    38
Total Spent       40
dtype: int64


In [8]:
# Build a price list from rows where Item and Price are both known
price_lookup = df.dropna(subset=["Item", "Price Per Unit"]).groupby("Item")["Price Per Unit"].first()
print(price_lookup)

# Fill missing Price Per Unit using the Item
m = df["Price Per Unit"].isna() & df["Item"].notna()
print("Price Per Unit recovered from Item:", m.sum())
df.loc[m, "Price Per Unit"] = df.loc[m, "Item"].map(price_lookup)

print("Price Per Unit still missing:", df["Price Per Unit"].isna().sum())

Item
Cake        3.0
Coffee      2.0
Cookie      1.0
Juice       3.0
Salad       5.0
Sandwich    4.0
Smoothie    4.0
Tea         1.5
Name: Price Per Unit, dtype: float64
Price Per Unit recovered from Item: 32
Price Per Unit still missing: 6


In [9]:
# Second pass: Total Spent = Quantity x Price Per Unit
m = df["Total Spent"].isna() & df["Quantity"].notna() & df["Price Per Unit"].notna()
print("Total Spent recovered:", m.sum())
df.loc[m, "Total Spent"] = df.loc[m, "Quantity"] * df.loc[m, "Price Per Unit"]

# Second pass: Quantity = Total Spent / Price Per Unit
m = df["Quantity"].isna() & df["Total Spent"].notna() & df["Price Per Unit"].notna()
print("Quantity recovered:", m.sum())
df.loc[m, "Quantity"] = df.loc[m, "Total Spent"] / df.loc[m, "Price Per Unit"]

print(df[["Quantity", "Price Per Unit", "Total Spent"]].isna().sum())

Total Spent recovered: 17
Quantity recovered: 15
Quantity          23
Price Per Unit     6
Total Spent       23
dtype: int64


In [10]:
# Prices that belong to only one item
unique_price_item = {1.0: "Cookie", 1.5: "Tea", 2.0: "Coffee", 5.0: "Salad"}

m = df["Item"].isna() & df["Price Per Unit"].isin(unique_price_item.keys())
print("Item recovered from Price:", m.sum())
df.loc[m, "Item"] = df.loc[m, "Price Per Unit"].map(unique_price_item)

print("Item still missing:", df["Item"].isna().sum())

Item recovered from Price: 489
Item still missing: 480


In [11]:
rows_before = len(df)
df = df.dropna(subset=["Quantity", "Price Per Unit", "Total Spent"])
print("Rows removed:", rows_before - len(df))
print("Rows remaining:", len(df))
print(df[["Quantity", "Price Per Unit", "Total Spent"]].isna().sum())

Rows removed: 26
Rows remaining: 9974
Quantity          0
Price Per Unit    0
Total Spent       0
dtype: int64


In [12]:

text_cols = ["Item", "Payment Method", "Location"]
print("Missing before:")
print(df[text_cols].isna().sum())

for col in text_cols:
    df[col] = df[col].fillna("Unknown")

print("Missing after:")
print(df.isna().sum())

Missing before:
Item               474
Payment Method    3168
Location          3952
dtype: int64
Missing after:
Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    460
dtype: int64


In [13]:
df["Quantity"] = df["Quantity"].astype(int)
print(df.dtypes)

Transaction ID                str
Item                          str
Quantity                    int64
Price Per Unit            float64
Total Spent               float64
Payment Method                str
Location                      str
Transaction Date    datetime64[s]
dtype: object


In [14]:
for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n = ((df[col] < lower) | (df[col] > upper)).sum()
    print(col, "| Q1:", q1, "Q3:", q3, "lower:", lower, "upper:", upper, "outliers:", n, "| min:", df[col].min(), "max:", df[col].max())

Quantity | Q1: 2.0 Q3: 4.0 lower: -1.0 upper: 7.0 outliers: 0 | min: 1 max: 5
Price Per Unit | Q1: 2.0 Q3: 4.0 lower: -1.0 upper: 7.0 outliers: 0 | min: 1.0 max: 5.0
Total Spent | Q1: 4.0 Q3: 12.0 lower: -8.0 upper: 24.0 outliers: 268 | min: 1.0 max: 25.0


In [15]:
before_nulls = df_raw.isna().sum().sum()
before_placeholders = (df_raw == "ERROR").sum().sum() + (df_raw == "UNKNOWN").sum().sum()
after_placeholders = (df == "ERROR").sum().sum() + (df == "UNKNOWN").sum().sum()

summary = pd.DataFrame({
    "Metric": [
        "Row count",
        "Blank (null) cells",
        "ERROR/UNKNOWN placeholder cells",
        "Total missing/invalid cells",
        "Duplicate rows",
        "Cells labelled 'Unknown' (deliberate)",
        "Columns with correct data type",
    ],
    "Before": [
        len(df_raw), before_nulls, before_placeholders,
        before_nulls + before_placeholders, df_raw.duplicated().sum(), 0, "4 of 8",
    ],
    "After": [
        len(df), df.isna().sum().sum(), after_placeholders,
        df.isna().sum().sum() + after_placeholders, df.duplicated().sum(),
        (df == "Unknown").sum().sum(), "8 of 8",
    ],
})
summary

,Metric,Before,After
0,Row count,10000,9974
1,Blank (null) cells,6826,460
2,ERROR/UNKNOWN placeholder cells,3256,0
3,Total missing/invalid cells,10082,460
4,Duplicate rows,0,0
5,Cells labelled 'Unknown' (deliberate),0,7594
6,Columns with correct data type,4 of 8,8 of 8


In [16]:
df.to_csv("cleaned_cafe_sales.csv", index=False)

check = pd.read_csv("cleaned_cafe_sales.csv")
print(check.shape)
print(check.isna().sum())

(9974, 8)
Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    460
dtype: int64


## Data Cleaning Decisions

*Dataset:* Cafe Sales - Dirty Data for Cleaning Training (Kaggle, Ahmed Mohamed). 10,000 rows, 8 columns.

| Issue found | Action taken | Why |
|---|---|---|
| 3,256 ERROR / UNKNOWN placeholder cells | Converted to proper missing values (NaN) | They are not real values, and isna() could not see them |
| Quantity, Price Per Unit, Total Spent stored as text | Converted to numbers | Needed for calculations |
| Transaction Date stored as Excel serial numbers (e.g. 45177) | Converted to real dates (2023-01-01 to 2023-12-31) | Needed for time analysis |
| Missing Total Spent, Quantity, Price Per Unit | Recalculated using Total Spent = Quantity x Price Per Unit (true in all 8,544 complete rows). First pass: 462 + 441 + 495 values | More accurate than guessing or using averages |
| Missing Price Per Unit | Filled from Item (each item has exactly one price): 32 values. Second calculation pass recovered 17 Total Spent + 15 Quantity | Item-price link is consistent in the data |
| Missing Item | Filled where the price belongs to one item only (1 = Cookie, 1.5 = Tea, 2 = Coffee, 5 = Salad): 489 values. Prices 3 and 4 are shared, so not guessed | Avoids wrong guesses |
| 26 rows with a numeric value that could not be recovered | Dropped (10,000 to 9,974 rows) | Only 0.26 percent of data; guessing sales amounts would add false numbers |
| Missing Item (474), Payment Method (3,168), Location (3,952) | Labelled Unknown | Cannot be worked out; dropping would lose about 40 percent of rows; most-common-value filling would distort the data |
| Missing Transaction Date (460) | Left blank | A date cannot be guessed; dropping would lose good sales data |
| Duplicate rows | 0 found, none removed | Transaction IDs are all unique |
| Outliers (IQR method) | Quantity 0, Price Per Unit 0, Total Spent 268: all kept | All 268 are 5 Salads at 5 each = 25, the largest valid order |
| Quantity stored as decimal | Converted to whole numbers | Quantity is a count |

*Result:* missing or invalid cells reduced from 10,082 to 460, and all 8 columns now have the correct data type